In [1]:
import pandas as pd

# File paths
input_file = 'test_tag_columns.csv'
output_file = 'transformed_text_medicine.csv'

# 1. Read the CSV file directly from the workplace
df = pd.read_csv(input_file)

# 2. Select only 'text' and 'Medicine' columns
filtered_df = df[['text', 'Medicine']]

# 3. Save the filtered dataset to a new CSV file
filtered_df.to_csv(output_file, index=False, encoding='utf-8-sig')

# 4. Preview the results
print("--- Data Transformed Successfully ---")
print(filtered_df.head())
print(f"\nSaved to: {output_file}")
print(f"Total Rows: {filtered_df.shape[0]} | Total Columns: {filtered_df.shape[1]}")

--- Data Transformed Successfully ---
                                                text  Medicine
0  আমি < NAME > । আমার বয়স 27 বছর । সাম্প্রতিক স...       NaN
1  আপনার প্রশ্নের জন্য ধন্যবাদ । দুশ্চিন্তা কমান ...  স্যালাইন
2  Thank you for your question . Your serum Trigl...       NaN
3  Proshno korar jonno dhonnobad . Ei boyosher ba...       NaN
4  বেশ কয়েকদিন যাবত গলায় খুব ব্যাথা সেই সাথে কাশি...       NaN

Saved to: transformed_text_medicine.csv
Total Rows: 3179 | Total Columns: 2


In [2]:
!pip install -q openai tqdm

import time
import pandas as pd
from tqdm import tqdm
from google.colab import userdata
from openai import OpenAI

# 1. Fetch API key from Colab Secrets
try:
    api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
    raise ValueError("Key 'OPENROUTER_API_KEY' not found in Colab Secrets. Please check the Secrets tab.") from e

# 2. Initialize OpenAI client configured for OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

# Set model name (adjust prefix if required by OpenRouter for this specific model)
MODEL_NAME = "gpt-5.6-luna"

# 3. Prompt Template
PROMPT_TEMPLATE = """You are given a Bangla medical sentence containing one or more medicine entities.

Your task is to create a modified version of the sentence by replacing exactly ONE medicine entity with a different but closely related medicine.

Rules:
1. Identify the specified medicine entity in the sentence.
2. Replace exactly ONE occurrence of that medicine with another medicine.
3. The replacement must be a different medicine from the original medicine.
4. The replacement must NOT be another medicine already present in the original sentence.
5. The replacement should be pharmacologically or therapeutically similar to the original medicine and should be plausible in the same medical context.
6. Prefer a medicine from the same therapeutic class, drug class, or with a closely related clinical use as the original medicine.
7. Do NOT replace the medicine with a generic or unrelated medicine simply because it is commonly used.
8. The replacement should fit naturally into the surrounding sentence without making the sentence medically or linguistically implausible.
9. Do not add any additional medicine.
10. Do not remove, add, or modify any other information in the sentence.
11. Keep the ENTIRE sentence structure intact. Do NOT truncate, cut short, or summarize any part of the original text.
12. Do not modify the original NER annotation.
13. Output ONLY the complete modified Bangla sentence from start to finish.
14. Do not provide explanations or identify the replacement.

Example:

Original sentence:
রোগীকে প্যারাসিটামল জ্বর কমানোর জন্য দেওয়া হয়েছে।

Medicine entity:
প্যারাসিটামল

Output:
রোগীকে আইবুপ্রোফেন জ্বর কমানোর জন্য দেওয়া হয়েছে।

Now perform the replacement.

Original sentence:
{SENTENCE}

Medicine entity:
{MEDICINE_ENTITY}

Modified sentence:"""

# 4. Load the filtered dataset
input_file = "transformed_text_medicine.csv"
output_file = "transformed_with_luna_modified.csv"

df = pd.read_csv(input_file)

# 5. Helper function to process individual rows
def get_modified_sentence(sentence, medicine_entity):
    if pd.isna(medicine_entity) or str(medicine_entity).strip() == "":
        return None  # Skip if no medicine entity is present

    prompt = PROMPT_TEMPLATE.format(
        SENTENCE=str(sentence).strip(),
        MEDICINE_ENTITY=str(medicine_entity).strip()
    )

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"\nError processing entity '{medicine_entity}': {e}")
        return None

# 6. Iterate through DataFrame and perform replacement
modified_sentences = []

print(f"Processing {len(df)} rows using {MODEL_NAME} via OpenRouter...")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    sentence = row['text']
    medicine = row['Medicine']

    modified_text = get_modified_sentence(sentence, medicine)
    modified_sentences.append(modified_text)

    # Optional small delay to respect API rate limits
    time.sleep(0.1)

# 7. Add results to DataFrame and export
df['modified_text'] = modified_sentences

# Save output
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\nProcessing complete! File saved as '{output_file}'.")

Processing 3179 rows using gpt-5.6-luna via OpenRouter...


100%|██████████| 3179/3179 [1:10:02<00:00,  1.32s/it]


Processing complete! File saved as 'transformed_with_luna_modified.csv'.


In [4]:
import pandas as pd

# Load the output file from your current run
df_final = pd.read_csv("transformed_with_luna_modified.csv")

# Drop rows where 'Medicine' or 'modified_text' is missing
df_clean = df_final.dropna(subset=['Medicine', 'modified_text']).copy()

# Save as clean dataset
df_clean.to_csv("transformed_with_luna_modified_cleaned.csv", index=False, encoding='utf-8-sig')

print(f"Original rows: {len(df_final)} | Cleaned rows: {len(df_clean)}")

Original rows: 3179 | Cleaned rows: 1058
